# Logit CompStat-Rio — Previsão de Roubo por Micropolígono

Implementa as Fases 1–4 do `06_prompt_claude_code_logit.md`.  
**Unidade espacial:** H3 res 9 (~0,1 km²) dentro das áreas da FM.  
**Unidade temporal:** semana ISO (`YYYY-WNN`). O campo `data` está válido em 99.4% dos registros (DD/MM/YYYY).  
**Horizontes:** s = 1, 2, 4 semanas à frente.  
**FM polygons:** convex hull das câmeras + buffer 120 m (sem `sh_area_forca`).

In [106]:
# ── Imports (todos aqui — nunca repetir em células seguintes) ─────────────────
import pandas as pd
import numpy as np
import json
import os
import warnings
from pathlib import Path

# Geospatial
import h3  # v4 API: latlng_to_cell, geo_to_cells, grid_disk, cell_to_latlng
from shapely.geometry import MultiPoint, Point, Polygon
from shapely.geometry import mapping as shapely_mapping
from shapely import wkt as shapely_wkt
from pyproj import Transformer
import geopandas as gpd

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Spatial statistics (Moran's I)
import libpysal.weights as lw
import esda

# Statsmodels (erros-padrão clusterizados)
import statsmodels.api as sm

# Visualization
import folium
from folium.plugins import HeatMap
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

Imports OK


In [107]:
# ── Configuração global ───────────────────────────────────────────────────────
DATA_DIR   = Path('../dados')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

RIO_BB   = dict(lat_min=-23.10, lat_max=-22.74, lon_min=-43.80, lon_max=-43.09)
RES      = 9      # resolução H3
BUFFER_M = 120    # buffer de borda em metros
HORIZONS = [1, 2, 4]  # horizontes em SEMANAS

# Feriados nacionais + municipais Rio 2020-2024 → convertidos para semana ISO
_HOLIDAY_DATES = [
    '2020-01-01','2020-02-24','2020-02-25','2020-03-01','2020-04-10','2020-04-21',
    '2020-05-01','2020-06-11','2020-09-07','2020-10-12','2020-11-02','2020-11-15','2020-12-25',
    '2021-01-01','2021-02-15','2021-02-16','2021-03-01','2021-04-02','2021-04-21',
    '2021-05-01','2021-06-03','2021-09-07','2021-10-12','2021-11-02','2021-11-15','2021-12-25',
    '2022-01-01','2022-02-28','2022-03-01','2022-04-15','2022-04-21',
    '2022-05-01','2022-06-16','2022-09-07','2022-10-12','2022-11-02','2022-11-15','2022-12-25',
    '2023-01-01','2023-02-20','2023-02-21','2023-03-01','2023-04-07','2023-04-21',
    '2023-05-01','2023-06-08','2023-09-07','2023-10-12','2023-11-02','2023-11-15','2023-12-25',
    '2024-01-01','2024-02-12','2024-02-13','2024-03-01','2024-03-29','2024-04-21',
    '2024-05-01','2024-05-30','2024-09-07','2024-10-12','2024-11-02','2024-11-15','2024-12-25',
]
_hd = pd.to_datetime(_HOLIDAY_DATES)
HOLIDAY_WEEKS = set(
    f"{d.isocalendar().year}-W{d.isocalendar().week:02d}" for d in _hd
)
print(f'Config OK | {len(HOLIDAY_WEEKS)} semanas com feriado')


# ── Dicionário de rótulos legíveis para policy makers ────────────────────────
# Converte nomes técnicos das features (n_fat_calcada, y_lag1, …) em descrições
# em português natural usadas nos mapas, popups e tabelas de drivers.

FEATURE_LABELS = {
    # Histórico do próprio hex
    "y_lag1":         "Houve crime na semana anterior",
    "y_lag2":         "Houve crime há 2 semanas",
    "y_lag4":         "Houve crime há 4 semanas",
    "y_lag12":        "Houve crime há 12 semanas",
    "n_crimes_12w":   "Crimes acumulados nas últimas 12 semanas",
    # Lag espacial
    "W_y_lag1":       "Crime na vizinhança imediata (semana anterior)",
    # Fatores urbanos (mapeados em cell-10-fatores)
    "n_fat_total":    "Fatores urbanos: total na área",
    "n_fat_ilum":     "Iluminação deficiente",
    "n_fat_vegetac":  "Vegetação obstruindo",
    "n_fat_lixo":     "Lixo / entulho acumulado",
    "n_fat_calcada":  "Calçada obstruída",
    "n_fat_transito": "Retenção de tráfego",
    "n_fat_mobil":    "Mobiliário urbano comprometido",
    "n_fat_sitrua":   "Situação de rua",
    # Policiamento
    "n_cameras":       "Câmeras no segmento",
    "n_cameras_ring1": "Câmeras na vizinhança imediata",
    # Sazonalidade temporal
    "week_sin":         "Sazonalidade semanal (ciclo anual)",
    "week_cos":         "Sazonalidade semanal (ciclo anual)",
    "is_holiday_week":  "Semana de feriado",
    # Denúncias
    "n_dd_lag1":      "Denúncias na semana anterior",
}

# Prefixos: tratamento dinâmico (orcrim_<faccao>, fe_area_<nome>)
def pretty_label(feat_name: str) -> str:
    """Retorna rótulo legível para um nome técnico de feature."""
    if feat_name in FEATURE_LABELS:
        return FEATURE_LABELS[feat_name]
    if feat_name.startswith("orcrim_"):
        faccao = feat_name.removeprefix("orcrim_")
        if faccao.lower() in ("sem domínio", "sem dominio"):
            return "Sem domínio territorial identificado"
        return f"Domínio territorial: {faccao}"
    if feat_name.startswith("fe_area_"):
        return "Efeito fixo da área FM"
    # Fallback: retorna o nome cru
    return feat_name


Config OK | 58 semanas com feriado


---
## Fase 1 — Exploração e entendimento dos dados

In [108]:
# ── Carrega todas as bases ────────────────────────────────────────────────────

# 1. Ocorrências criminais — RÓTULO do modelo
df_oc = pd.read_csv(
    DATA_DIR / 'df_ocorrencias_tratado - Extração 1 .csv',
    encoding='utf-8', low_memory=False
)

# 2. Câmeras / Polígonos FM — define as áreas e contagem de câmeras por hex
df_cam = pd.read_csv(DATA_DIR / 'cameras_areas_fm.csv', encoding='utf-8')
coords = df_cam['geometry'].str.extract(r'POINT \(([^ ]+) ([^\)]+)\)')
df_cam['lon'] = coords[0].astype(float)
df_cam['lat'] = coords[1].astype(float)

# 3. Fatores urbanos — desordem (estático, sem campo de data)
df_fat = pd.read_csv(DATA_DIR / 'fatores_urbanos.csv', encoding='utf-8')
# coordenada_x = latitude, coordenada_y = longitude (confirmado pelo range de valores)
df_fat.rename(columns={'coordenada_x': 'latitude', 'coordenada_y': 'longitude'}, inplace=True)

# 4. Disque Denúncia — dinâmica criminal (lat/lon com vírgula decimal)
df_dd = pd.read_csv(DATA_DIR / 'disk_denuncia.csv', encoding='latin1', sep=';', low_memory=False)
df_dd['lat_dd'] = pd.to_numeric(df_dd['latitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_dd['lon_dd'] = pd.to_numeric(df_dd['longitude'].astype(str).str.replace(',', '.'), errors='coerce')
df_dd['date_dd'] = pd.to_datetime(df_dd['data_denuncia'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
# ISO week is computed in cell-13 from date_dd directly

# 5. Domínio territorial — contexto ORCRIM (estático)
df_dom = pd.read_csv(
    DATA_DIR / 'outros dados' / 'dominio_territorial - Extração 1.csv',
    encoding='utf-8'
)

print('=== SCHEMA RESUMIDO DAS BASES ===')
print(f'\n[Ocorrências] {df_oc.shape} | colunas: {list(df_oc.columns)}')
print(f'  data válida (não-nula): {df_oc["data"].notna().sum():,} / {len(df_oc):,}')
print(f'  ano range: {df_oc["ano"].min()} – {df_oc["ano"].max()}')
print(f'  mes range: {df_oc["mes"].min()} – {df_oc["mes"].max()}')
print(f'  lat range: {df_oc["latitude"].min():.3f} – {df_oc["latitude"].max():.3f}')

print(f'\n[Câmeras FM] {df_cam.shape} | áreas: {df_cam["nome_area_fm"].nunique()}')
print(f'  áreas: {df_cam["nome_area_fm"].unique().tolist()}')

print(f'\n[Fatores Urbanos] {df_fat.shape} | tipos: {df_fat["tipo_ocorrencia_descricao"].nunique()}')
print(f'  top tipos:\n{df_fat["tipo_ocorrencia_descricao"].value_counts().head(5)}')

print(f'\n[Disque Denúncia] {df_dd.shape} | lat válida: {df_dd["lat_dd"].notna().sum():,} | date válida: {df_dd["date_dd"].notna().sum():,}')

print(f'\n[Domínio Territorial] {df_dom.shape} | facções: {df_dom["dominio_orcrim"].unique()}')

=== SCHEMA RESUMIDO DAS BASES ===

[Ocorrências] (115354, 14) | colunas: ['id_criptografado', 'ano', 'data', 'mes', 'hora', 'delito', 'longitude', 'latitude', 'desc_delito', 'aisp', 'risp', 'locf', 'dia_semana', 'geometria']
  data válida (não-nula): 115,332 / 115,354
  ano range: 2020 – 2024
  mes range: 1 – 12
  lat range: -22901.000 – -22.784

[Câmeras FM] (985, 6) | áreas: 9
  áreas: ['Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia', 'Rodoviária - Terminal Gentileza - Estação Leopoldina', 'Rua Lauro Müller – Avenida General Severiano – Avenida Venceslau Brás', 'Praia de Botafogo - Rua Marquês de Abrantes', 'Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria', 'Jardim de Alah', 'Estações São Francisco Xavier - Afonso Pena', 'Bangu: Calçadão - Bangu Shopping', 'Campo Grande: Estação de Trem - Calçadão']

[Fatores Urbanos] (2085, 31) | tipos: 22
  top tipos:
tipo_ocorrencia_descricao
Vegetação encobrindo iluminação pública           327
Pessoas em sit

---
## Fase 2 — Construção do painel hex × mês

In [109]:
# ── 2.1 Polígonos das áreas FM (convex hull das câmeras + buffer 120m) ────────
# Nota: arquivo sh_area_forca não encontrado em dados/. Usando convex hull das
# câmeras como aproximação do polígono operacional de cada área.

# Transformadores UTM-23S ↔ WGS84 para buffer métrico preciso
t_fwd = Transformer.from_crs('EPSG:4326', 'EPSG:31983', always_xy=True)  # WGS84 → UTM
t_inv = Transformer.from_crs('EPSG:31983', 'EPSG:4326', always_xy=True)  # UTM → WGS84

def build_fm_polygon_buffered(cam_df, buffer_m=120):
    """Convex hull dos pontos de câmera + buffer métrico em EPSG:31983."""
    points = MultiPoint([(r.lon, r.lat) for _, r in cam_df.iterrows()])
    hull = points.convex_hull
    # Reprojetar para UTM-23S, aplicar buffer, reprojetar de volta
    coords_utm = [t_fwd.transform(x, y) for x, y in hull.exterior.coords]
    hull_utm   = Polygon(coords_utm).buffer(buffer_m)
    coords_wgs = [t_inv.transform(x, y) for x, y in hull_utm.exterior.coords]
    return Polygon(coords_wgs)

fm_polygons = {}  # area_name → shapely Polygon
for area, grp in df_cam.groupby('nome_area_fm'):
    fm_polygons[area] = build_fm_polygon_buffered(grp, buffer_m=BUFFER_M)
    print(f'  {area[:55]:<55} | {grp.shape[0]:>3} câmeras')

print(f'\nTotal de áreas FM: {len(fm_polygons)}')

  Bangu: Calçadão - Bangu Shopping                        |  30 câmeras
  Campo Grande: Estação de Trem - Calçadão                |  45 câmeras
  Estações São Francisco Xavier - Afonso Pena             |  60 câmeras
  Jardim de Alah                                          |  30 câmeras
  Metrô Botafogo - Rua São Clemente - Rua Voluntários da  |  80 câmeras
  Praia de Botafogo - Rua Marquês de Abrantes             | 150 câmeras
  Presidente Vargas - Campo de Santana - Central do Brasi | 230 câmeras
  Rodoviária - Terminal Gentileza - Estação Leopoldina    | 310 câmeras
  Rua Lauro Müller – Avenida General Severiano – Avenida  |  50 câmeras

Total de áreas FM: 9


In [110]:
# ── 2.2 Universo de hexágonos H3 res 9 por área FM ───────────────────────────
# geo_to_cells (h3 v4) aceita GeoJSON polygon dict

rows_hex = []
for area_name, poly in fm_polygons.items():
    geojson_poly = shapely_mapping(poly)  # GeoJSON dict
    cells = h3.geo_to_cells(geojson_poly, RES)
    for cell in cells:
        lat_c, lon_c = h3.cell_to_latlng(cell)
        rows_hex.append({'hex_id': cell, 'area_fm': area_name,
                         'lat_centroide': lat_c, 'lon_centroide': lon_c})

df_fm = pd.DataFrame(rows_hex)

# Conflito de borda: hex atribuído à área cujo centroide de câmera é mais próximo
area_centroids = {}  # area_name → (lat, lon) do centroide médio das câmeras
for area, grp in df_cam.groupby('nome_area_fm'):
    area_centroids[area] = (grp['lat'].mean(), grp['lon'].mean())

# Para hexágonos duplicados, manter o de área mais próxima
def nearest_area(row, centroids):
    dists = {a: np.sqrt((row.lat_centroide - c[0])**2 + (row.lon_centroide - c[1])**2)
             for a, c in centroids.items()}
    return min(dists, key=dists.get)

duplicated_hexes = df_fm[df_fm.duplicated('hex_id', keep=False)]['hex_id'].unique()
if len(duplicated_hexes) > 0:
    mask_dup = df_fm['hex_id'].isin(duplicated_hexes)
    df_fm.loc[mask_dup, 'area_fm'] = df_fm[mask_dup].apply(
        lambda r: nearest_area(r, area_centroids), axis=1
    )

df_fm = df_fm.drop_duplicates('hex_id').reset_index(drop=True)
fm_hex_set = set(df_fm['hex_id'])

print(f'Total hexágonos FM res {RES}: {len(df_fm)}')
print(df_fm.groupby('area_fm')['hex_id'].count().rename('n_hex').sort_values())

Total hexágonos FM res 9: 156
area_fm
Jardim de Alah                                                            6
Bangu: Calçadão - Bangu Shopping                                          7
Rua Lauro Müller – Avenida General Severiano – Avenida Venceslau Brás    10
Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria            13
Campo Grande: Estação de Trem - Calçadão                                 19
Estações São Francisco Xavier - Afonso Pena                              19
Praia de Botafogo - Rua Marquês de Abrantes                              20
Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia    25
Rodoviária - Terminal Gentileza - Estação Leopoldina                     37
Name: n_hex, dtype: int64


In [111]:
# ── 2.3 Índice temporal: semana ISO (2020-W01 → 2024-W52) ────────────────────
# O campo 'data' tem 99.4% de registros válidos (DD/MM/YYYY) dentro de 2020-2024.
# Unidade temporal: semana ISO → ~262 semanas, compatível com cadência CompStat.

all_mondays = pd.date_range('2020-01-06', '2024-12-30', freq='W-MON')
df_weeks = pd.DataFrame({'monday': all_mondays})
_iso = df_weeks['monday'].map(lambda d: d.isocalendar())
df_weeks['iso_year'] = _iso.map(lambda x: x.year)
df_weeks['iso_week'] = _iso.map(lambda x: x.week)
df_weeks['ano_sem']  = df_weeks.apply(lambda r: f"{r.iso_year}-W{r.iso_week:02d}", axis=1)
df_weeks['t'] = range(len(df_weeks))

# Sazonalidade semanal (sin/cos)
df_weeks['week_sin'] = np.sin(2 * np.pi * df_weeks['iso_week'] / 52)
df_weeks['week_cos'] = np.cos(2 * np.pi * df_weeks['iso_week'] / 52)
df_weeks['is_holiday_week'] = df_weeks['ano_sem'].isin(HOLIDAY_WEEKS).astype(int)

print(f'Semanas: {df_weeks["ano_sem"].iloc[0]} → {df_weeks["ano_sem"].iloc[-1]}')
print(f'Total: {len(df_weeks)} semanas | {df_weeks["is_holiday_week"].sum()} com feriado')

Semanas: 2020-W02 → 2025-W01
Total: 261 semanas | 57 com feriado


In [112]:
# ── 2.4 Rótulo: y = 1 se ≥ 1 roubo no hex na semana ─────────────────────────

# Filtra ocorrências com data válida no Rio
df_oc['data_parsed'] = pd.to_datetime(df_oc['data'], dayfirst=True, errors='coerce')
df_oc_valid = df_oc[
    df_oc['data_parsed'].dt.year.between(2020, 2024) &
    df_oc['latitude'].between(RIO_BB['lat_min'], RIO_BB['lat_max']) &
    df_oc['longitude'].between(RIO_BB['lon_min'], RIO_BB['lon_max'])
].copy()
print(f'Ocorrências válidas (data + bbox): {len(df_oc_valid):,}')

# Semana ISO de cada ocorrência
_iso_oc = df_oc_valid['data_parsed'].map(lambda d: d.isocalendar())
df_oc_valid['ano_sem'] = df_oc_valid['data_parsed'].apply(
    lambda d: f"{d.isocalendar().year}-W{d.isocalendar().week:02d}"
)

# Atribui hex_id
df_oc_valid['hex_id'] = [
    h3.latlng_to_cell(float(lat), float(lon), RES)
    for lat, lon in zip(df_oc_valid['latitude'], df_oc_valid['longitude'])
]

# Mantém apenas crimes dentro da FM
df_oc_fm = df_oc_valid[df_oc_valid['hex_id'].isin(fm_hex_set)].copy()
print(f'Ocorrências dentro da FM: {len(df_oc_fm):,} ({len(df_oc_fm)/len(df_oc_valid):.1%} do total)')
print(f'\nDistribuição por tipo:\n{df_oc_fm["desc_delito"].value_counts()}')

# Contagem por hex × semana
crimes_hex_sem = (
    df_oc_fm.groupby(['hex_id', 'ano_sem'])
    .size()
    .reset_index(name='n_crimes')
)

# Painel completo: produto cartesiano hex × semana
df_panel = (
    df_fm[['hex_id', 'area_fm', 'lat_centroide', 'lon_centroide']]
    .merge(df_weeks[['ano_sem', 't', 'week_sin', 'week_cos', 'is_holiday_week']], how='cross')
    .merge(crimes_hex_sem, on=['hex_id', 'ano_sem'], how='left')
)
df_panel['n_crimes'] = df_panel['n_crimes'].fillna(0).astype(int)
df_panel['y'] = (df_panel['n_crimes'] >= 1).astype(int)

print(f'\nPainel: {len(df_panel):,} linhas | {df_fm["hex_id"].nunique()} hex × {len(df_weeks)} semanas')
print(f'Taxa de positivos global: {df_panel["y"].mean():.1%}')
print('\nPor área FM:')
print(df_panel.groupby('area_fm')['y'].mean().round(3).sort_values())

Ocorrências válidas (data + bbox): 114,584
Ocorrências dentro da FM: 15,964 (13.9% do total)

Distribuição por tipo:
desc_delito
Roubo a transeunte           9775
Roubo de aparelho celular    4696
Roubo em coletivo            1493
Name: count, dtype: int64

Painel: 40,716 linhas | 156 hex × 261 semanas
Taxa de positivos global: 24.8%

Por área FM:
area_fm
Campo Grande: Estação de Trem - Calçadão                                 0.065
Praia de Botafogo - Rua Marquês de Abrantes                              0.181
Jardim de Alah                                                           0.185
Rodoviária - Terminal Gentileza - Estação Leopoldina                     0.194
Rua Lauro Müller – Avenida General Severiano – Avenida Venceslau Brás    0.204
Bangu: Calçadão - Bangu Shopping                                         0.210
Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria            0.247
Estações São Francisco Xavier - Afonso Pena                              0.251
Presidente

In [113]:
# ── 2.5 Lags do rótulo ────────────────────────────────────────────────────────
# Ordenado por hex + t para garantir lag correto

df_panel = df_panel.sort_values(["hex_id", "t"]).reset_index(drop=True)

# Lags próprios (shift dentro de cada hex)
for lag in [1, 2, 4, 12]:
    df_panel[f"y_lag{lag}"] = df_panel.groupby("hex_id")["y"].shift(lag).fillna(0).astype(int)

# Janela acumulada de 12 semanas (contagem de crimes nas últimas 12 obs)
df_panel["n_crimes_12w"] = (
    df_panel.groupby("hex_id")["n_crimes"]
    .transform(lambda s: s.shift(1).rolling(12, min_periods=1).sum())
    .fillna(0)
)

print('Lags calculados:')
print(df_panel[["hex_id", "ano_sem", "y", "y_lag1", "y_lag2", "y_lag4", "y_lag12", "n_crimes_12w"]].head(15))

Lags calculados:
             hex_id   ano_sem  y  y_lag1  y_lag2  y_lag4  y_lag12  \
0   89a8a029027ffff  2020-W02  0       0       0       0        0   
1   89a8a029027ffff  2020-W03  0       0       0       0        0   
2   89a8a029027ffff  2020-W04  0       0       0       0        0   
3   89a8a029027ffff  2020-W05  0       0       0       0        0   
4   89a8a029027ffff  2020-W06  0       0       0       0        0   
5   89a8a029027ffff  2020-W07  0       0       0       0        0   
6   89a8a029027ffff  2020-W08  0       0       0       0        0   
7   89a8a029027ffff  2020-W09  0       0       0       0        0   
8   89a8a029027ffff  2020-W10  0       0       0       0        0   
9   89a8a029027ffff  2020-W11  0       0       0       0        0   
10  89a8a029027ffff  2020-W12  0       0       0       0        0   
11  89a8a029027ffff  2020-W13  0       0       0       0        0   
12  89a8a029027ffff  2020-W14  0       0       0       0        0   
13  89a8a029027ff

In [114]:
# ── 2.6 Lag espacial W_y_lag1 (vizinhos ring-1 dentro da FM) ─────────────────
# W_y_lag1[h,t] = média de y_lag1 dos vizinhos ring-1 de h que estão na FM.
# Vizinhos fora da FM contribuem com 0 (não excluídos da média, per spec).

# Pré-computa vizinhos ring-1 para cada hex (estático, não muda por período)
# h3.grid_disk retorna list em v4, por isso convertemos para set antes de subtrair
hex_neighbors = {}
for cell in fm_hex_set:
    neighbors_ring1 = set(h3.grid_disk(cell, 1)) - {cell}
    hex_neighbors[cell] = neighbors_ring1

print('Computando lag espacial (pode levar ~30s)...')

spatial_lag_rows = []
for t_val, grp in df_panel.groupby("t"):
    # Índice hex_id → y_lag1 neste período
    y_lag1_map = grp.set_index("hex_id")["y_lag1"].to_dict()

    for _, row in grp.iterrows():
        neighbors = hex_neighbors.get(row["hex_id"], set())
        # Vizinhos FM: contribuem com seu y_lag1; vizinhos fora FM: contribuem com 0
        vals = [y_lag1_map.get(nb, 0) for nb in neighbors]
        w_lag = np.mean(vals) if vals else 0.0
        spatial_lag_rows.append({"hex_id": row["hex_id"], "t": t_val, "W_y_lag1": w_lag})

df_spatial = pd.DataFrame(spatial_lag_rows)
df_panel = df_panel.merge(df_spatial, on=["hex_id", "t"], how="left")
df_panel["W_y_lag1"] = df_panel["W_y_lag1"].fillna(0.0)

print(f"W_y_lag1 calculado | média: {df_panel["W_y_lag1"].mean():.3f} | max: {df_panel["W_y_lag1"].max():.3f}")


Computando lag espacial (pode levar ~30s)...
W_y_lag1 calculado | média: 0.191 | max: 1.000


In [115]:
# ── 2.7 Covariáveis de desordem urbana (fatores_urbanos — dataset estático) ──
# Fatores urbanos não têm campo de data → tratados como covariável estática por hex.
# Agregação: contagem de ocorrências de cada tipo por hex.

# Filtra fatores com coordenadas válidas
df_fat_valid = df_fat[
    df_fat['latitude'].between(RIO_BB['lat_min'], RIO_BB['lat_max']) &
    df_fat['longitude'].between(RIO_BB['lon_min'], RIO_BB['lon_max'])
].copy()

df_fat_valid['hex_id'] = [
    h3.latlng_to_cell(float(lat), float(lon), RES)
    for lat, lon in zip(df_fat_valid['latitude'], df_fat_valid['longitude'])
]

# Apenas fatores dentro da FM
df_fat_fm = df_fat_valid[df_fat_valid['hex_id'].isin(fm_hex_set)].copy()
print(f'Fatores dentro da FM: {len(df_fat_fm):,} de {len(df_fat_valid):,} válidos')

# Mapeamento de tipos relevantes para a literatura
TIPO_MAP = {
    'ilum':    ['Iluminação', 'luminosa', 'luz'],
    'vegetac': ['Vegetação', 'poda', 'vegeta'],
    'lixo':    ['Lixo', 'entulho', 'sujeira', 'limpeza'],
    'calcada': ['calçada', 'passeio', 'obstáculo', 'obstrução'],
    'transito':['trânsito', 'tráfego', 'retenção'],
    'mobil':   ['mobiliário', 'mobilia'],
    'sitrua':  ['situação de rua', 'moradores de rua'],
}

desc_col = 'tipo_ocorrencia_descricao'
for nome, keywords in TIPO_MAP.items():
    pattern = '|'.join(keywords)
    df_fat_fm[f'is_{nome}'] = df_fat_fm[desc_col].str.contains(pattern, case=False, na=False).astype(int)

# Agrega por hex (contagem total + por tipo)
fat_cols = [f'is_{k}' for k in TIPO_MAP]
df_fat_hex = (
    df_fat_fm.groupby('hex_id')[fat_cols]
    .sum()
    .rename(columns={c: c.replace('is_', 'n_fat_') for c in fat_cols})
    .reset_index()
)
df_fat_hex['n_fat_total'] = df_fat_hex[[c for c in df_fat_hex.columns if c.startswith('n_fat_')]].sum(axis=1)

print(f'Hexes com fatores urbanos: {len(df_fat_hex)}')
print(df_fat_hex[[c for c in df_fat_hex.columns if c != 'hex_id']].sum().sort_values(ascending=False))

Fatores dentro da FM: 905 de 2,085 válidos
Hexes com fatores urbanos: 91
n_fat_total       815
n_fat_vegetac     209
n_fat_calcada     178
n_fat_sitrua      150
n_fat_ilum        149
n_fat_transito     77
n_fat_mobil        40
n_fat_lixo         12
dtype: int64


In [116]:
# ── 2.8 Covariáveis de policiamento: câmeras por hex ─────────────────────────

df_cam["hex_id"] = [
    h3.latlng_to_cell(float(lat), float(lon), RES)
    for lat, lon in zip(df_cam["lat"], df_cam["lon"])
]

# Câmeras no próprio hex
df_cam_hex = df_cam.groupby("hex_id").size().reset_index(name="n_cameras")

# Câmeras no ring-1 (vizinhança imediata)
# h3.grid_disk retorna list em v4, por isso convertemos para set antes de subtrair
cam_count = df_cam["hex_id"].value_counts().to_dict()
df_cam_hex["n_cameras_ring1"] = df_cam_hex["hex_id"].apply(
    lambda h: sum(cam_count.get(nb, 0) for nb in set(h3.grid_disk(h, 1)) - {h})
)

print(f"Hexes com câmera: {len(df_cam_hex)} | câmeras total: {df_cam_hex["n_cameras"].sum()}")
print(df_cam_hex.describe().round(2))

Hexes com câmera: 141 | câmeras total: 985
       n_cameras  n_cameras_ring1
count     141.00           141.00
mean        6.99            30.73
std         5.93            22.19
min         1.00             1.00
25%         3.00            13.00
50%         5.00            23.00
75%         9.00            43.00
max        25.00            87.00


In [117]:
# ── 2.9 Contexto territorial ORCRIM (estático) ───────────────────────────────
# Spatial join: centroide de cada hex → polígono de domínio territorial

# Parseia geometrias WKT
dom_polygons = []
for _, row in df_dom.iterrows():
    try:
        geom = shapely_wkt.loads(row['geometria'])
        dom_polygons.append({'orcrim': row['dominio_orcrim'],
                              'nome_ter': row['nome_territorio'],
                              'geometry': geom})
    except Exception:
        continue

gdf_dom = gpd.GeoDataFrame(dom_polygons, crs='EPSG:4326')

# Converte centroides dos hexes FM para GeoDataFrame
gdf_hex = gpd.GeoDataFrame(
    df_fm[['hex_id']],
    geometry=gpd.points_from_xy(df_fm['lon_centroide'], df_fm['lat_centroide']),
    crs='EPSG:4326'
)

# Spatial join: ponto dentro do polígono ORCRIM
gdf_joined = gpd.sjoin(gdf_hex, gdf_dom[['orcrim', 'geometry']], how='left', predicate='within')

# Remove duplicatas (hex em mais de um polígono — raro)
df_orcrim_hex = (
    gdf_joined[['hex_id', 'orcrim']]
    .drop_duplicates('hex_id')
    .fillna({'orcrim': 'Sem domínio'})
)

print(f'ORCRIM por hex:\n{df_orcrim_hex["orcrim"].value_counts()}')

ORCRIM por hex:
orcrim
Sem domínio    146
Milícia          8
CV               1
TCP              1
Name: count, dtype: int64


In [118]:
# ── 2.10 Disque Denúncia: contagem semanal por hex ──────────────────────────

n_dd_valid = df_dd[["lat_dd", "lon_dd", "date_dd"]].dropna().shape[0]
print(f"DD com lat + date válidos: {n_dd_valid:,} / {len(df_dd):,}")

if n_dd_valid >= 100:
    df_dd_valid = df_dd[
        df_dd["lat_dd"].between(RIO_BB["lat_min"], RIO_BB["lat_max"]) &
        df_dd["lon_dd"].between(RIO_BB["lon_min"], RIO_BB["lon_max"]) &
        df_dd["date_dd"].notna()
    ].copy()

    df_dd_valid["hex_id"] = [
        h3.latlng_to_cell(float(lat), float(lon), RES)
        for lat, lon in zip(df_dd_valid["lat_dd"], df_dd_valid["lon_dd"])
    ]
    df_dd_fm = df_dd_valid[df_dd_valid["hex_id"].isin(fm_hex_set)].copy()

    # Agrega para semana ISO (mesmo padrão YYYY-WNN usado no painel)
    df_dd_fm["ano_sem"] = df_dd_fm["date_dd"].apply(
        lambda d: f"{d.isocalendar().year}-W{d.isocalendar().week:02d}" if pd.notna(d) else None
    )
    df_dd_fm = df_dd_fm.dropna(subset=["ano_sem"])

    df_dd_hex_sem = (
        df_dd_fm.groupby(["hex_id", "ano_sem"])
        .size()
        .reset_index(name="n_dd")
    )
    print(f"DD dentro da FM: {df_dd_fm.shape[0]:,} | hex-semanas únicas: {len(df_dd_hex_sem):,}")
    DD_AVAILABLE = True
else:
    print("DD sem lat/date suficientes → excluído do modelo (documentado)")
    df_dd_hex_sem = pd.DataFrame(columns=["hex_id", "ano_sem", "n_dd"])
    DD_AVAILABLE = False

DD com lat + date válidos: 17,850 / 83,549
DD dentro da FM: 1,243 | hex-semanas únicas: 1,168


In [119]:
# ── 2.11 Painel completo: merge de todas as features ─────────────────────────

# Base: painel hex × semana com lags e lag espacial
df_full = df_panel.copy()

# Fatores urbanos (estático — join por hex_id)
df_full = df_full.merge(df_fat_hex, on="hex_id", how="left")
fat_feat_cols = [c for c in df_fat_hex.columns if c != "hex_id"]
df_full[fat_feat_cols] = df_full[fat_feat_cols].fillna(0)

# Câmeras (estático — join por hex_id)
df_full = df_full.merge(df_cam_hex, on="hex_id", how="left")
df_full[["n_cameras", "n_cameras_ring1"]] = df_full[["n_cameras", "n_cameras_ring1"]].fillna(0)

# ORCRIM (estático — join por hex_id, codificado como dummy)
df_full = df_full.merge(df_orcrim_hex, on="hex_id", how="left")
df_full["orcrim"] = df_full["orcrim"].fillna("Sem domínio")
orcrim_dummies = pd.get_dummies(df_full["orcrim"], prefix="orcrim", drop_first=False)
df_full = pd.concat([df_full, orcrim_dummies], axis=1)

# Disque Denúncia (dinâmico — join por hex_id + ano_sem)
if DD_AVAILABLE:
    df_full = df_full.merge(df_dd_hex_sem, on=["hex_id", "ano_sem"], how="left")
    df_full["n_dd"] = df_full["n_dd"].fillna(0)
    # Lag 1 do DD para evitar simultaneidade
    df_full["n_dd_lag1"] = df_full.groupby("hex_id")["n_dd"].shift(1).fillna(0)
else:
    df_full["n_dd_lag1"] = 0

print(f"Painel final: {len(df_full):,} linhas × {len(df_full.columns)} colunas")
print(f"Positivos: {df_full["y"].sum():,} ({df_full["y"].mean():.1%})")
print(f"Colunas: {list(df_full.columns)}")

# Salva painel
df_full.to_csv(OUTPUT_DIR / "painel_hex_semana_res9.csv", index=False)
print(f"Salvo: output/painel_hex_semana_res9.csv")

Painel final: 40,716 linhas × 34 colunas
Positivos: 10,106 (24.8%)
Colunas: ['hex_id', 'area_fm', 'lat_centroide', 'lon_centroide', 'ano_sem', 't', 'week_sin', 'week_cos', 'is_holiday_week', 'n_crimes', 'y', 'y_lag1', 'y_lag2', 'y_lag4', 'y_lag12', 'n_crimes_12w', 'W_y_lag1', 'n_fat_ilum', 'n_fat_vegetac', 'n_fat_lixo', 'n_fat_calcada', 'n_fat_transito', 'n_fat_mobil', 'n_fat_sitrua', 'n_fat_total', 'n_cameras', 'n_cameras_ring1', 'orcrim', 'orcrim_CV', 'orcrim_Milícia', 'orcrim_Sem domínio', 'orcrim_TCP', 'n_dd', 'n_dd_lag1']
Salvo: output/painel_hex_semana_res9.csv


In [120]:
# ── 2.12 Verificação de esparsidade — Campo Grande ───────────────────────────
# Spec: se taxa de positivos < 10%, rodar em res 8 separadamente

sparsity = df_full.groupby('area_fm')['y'].agg(['mean', 'sum', 'count'])
sparsity.columns = ['taxa_positivos', 'n_positivos', 'n_obs']
sparsity = sparsity.sort_values('taxa_positivos')
display(sparsity.style.background_gradient(subset=['taxa_positivos'], cmap='RdYlGn'))

SPARSE_AREAS = sparsity[sparsity['taxa_positivos'] < 0.10].index.tolist()
if SPARSE_AREAS:
    print(f'\nÁreas esparsas (< 10% positivos): {SPARSE_AREAS}')
    print('→ Salvaguarda: class_weight="balanced" + monitorar AUC-PR na validação')
    print('  (res 8 para Campo Grande será ativado se AUC-PR < 0.10 na Fase 3)')
else:
    print('Nenhuma área com taxa < 10% — todas aptas para estimação direta')

,taxa_positivos,n_positivos,n_obs
area_fm,,,
Campo Grande: Estação de Trem - Calçadão,0.065336,324,4959
Praia de Botafogo - Rua Marquês de Abrantes,0.181418,947,5220
Jardim de Alah,0.184547,289,1566
Rodoviária - Terminal Gentileza - Estação Leopoldina,0.194160,1875,9657
Rua Lauro Müller – Avenida General Severiano – Avenida Venceslau Brás,0.204215,533,2610
Bangu: Calçadão - Bangu Shopping,0.210181,384,1827
Metrô Botafogo - Rua São Clemente - Rua Voluntários da Pátria,0.246979,838,3393
Estações São Francisco Xavier - Afonso Pena,0.251059,1245,4959
Presidente Vargas - Campo de Santana - Central do Brasil - Cinelândia,0.562605,3671,6525



Áreas esparsas (< 10% positivos): ['Campo Grande: Estação de Trem - Calçadão']
→ Salvaguarda: class_weight="balanced" + monitorar AUC-PR na validação
  (res 8 para Campo Grande será ativado se AUC-PR < 0.10 na Fase 3)


---
## Fase 3 — Estimação do logit por horizonte

In [121]:
# ── 3.1 Define grupos de features ────────────────────────────────────────────

# Features numéricas
FEAT_HIST    = ["y_lag1", "y_lag2", "y_lag4", "y_lag12", "n_crimes_12w"]
FEAT_SPATIAL = ["W_y_lag1"]
FEAT_FAT     = [c for c in df_full.columns if c.startswith("n_fat_")]
FEAT_POL     = ["n_cameras", "n_cameras_ring1"]
FEAT_TEMP    = ["week_sin", "week_cos", "is_holiday_week"]
FEAT_DD      = ["n_dd_lag1"] if DD_AVAILABLE else []
FEAT_ORCRIM  = [c for c in df_full.columns if c.startswith("orcrim_")]

NUMERIC_FEATS = FEAT_HIST + FEAT_SPATIAL + FEAT_FAT + FEAT_POL + FEAT_TEMP + FEAT_DD + FEAT_ORCRIM

# Efeito fixo de área FM (one-hot, drop_first para evitar multicolinearidade)
# codificado diretamente: cria dummies antes da estimação
area_dummies = pd.get_dummies(df_full["area_fm"], prefix="fe_area", drop_first=True)
df_full = pd.concat([df_full, area_dummies], axis=1)
FE_COLS = list(area_dummies.columns)

ALL_FEATS = NUMERIC_FEATS + FE_COLS

print(f"Total de features: {len(ALL_FEATS)}")
print(f"  Histórico próprio: {FEAT_HIST}")
print(f"  Lag espacial:      {FEAT_SPATIAL}")
print(f"  Fatores urbanos:   {FEAT_FAT}")
print(f"  Policiamento:      {FEAT_POL}")
print(f"  Temporal:          {FEAT_TEMP}")
print(f"  Disque Denúncia:   {FEAT_DD}")
print(f"  ORCRIM dummies:    {FEAT_ORCRIM}")
print(f"  Efeitos fixos FM:  {len(FE_COLS)} dummies")

Total de features: 32
  Histórico próprio: ['y_lag1', 'y_lag2', 'y_lag4', 'y_lag12', 'n_crimes_12w']
  Lag espacial:      ['W_y_lag1']
  Fatores urbanos:   ['n_fat_ilum', 'n_fat_vegetac', 'n_fat_lixo', 'n_fat_calcada', 'n_fat_transito', 'n_fat_mobil', 'n_fat_sitrua', 'n_fat_total']
  Policiamento:      ['n_cameras', 'n_cameras_ring1']
  Temporal:          ['week_sin', 'week_cos', 'is_holiday_week']
  Disque Denúncia:   ['n_dd_lag1']
  ORCRIM dummies:    ['orcrim_CV', 'orcrim_Milícia', 'orcrim_Sem domínio', 'orcrim_TCP']
  Efeitos fixos FM:  8 dummies


In [122]:
# ── 3.2 Estimação dos 3 modelos (s = 1, 2, 4 meses) ─────────────────────────
# Validação temporal: treina até T-4, valida T-3..T-1, testa em T.
# NUNCA usa split aleatório — painel tem dependência temporal.

t_max = df_full['t'].max()
T_TEST     = t_max          # período T (mais recente)
T_VAL_END  = t_max - 1     # validação: T-3 a T-1
T_VAL_START= t_max - 3
T_TRAIN_END= t_max - 4     # treino: até T-4

print(f't_max = {t_max} (período mais recente)')
print(f'Treino: t ≤ {T_TRAIN_END} | Validação: {T_VAL_START}–{T_VAL_END} | Teste: t = {T_TEST}')

# Descarta as primeiras 12 observações por hex (necessárias para lag12 válido)
df_est = df_full[df_full.groupby('hex_id')['t'].transform(lambda s: s - s.min()) >= 12].copy()
print(f'Obs após descartar lag12 warmup: {len(df_est):,}')

models = {}   # {s: fitted LogisticRegression}
coef_rows = []

for s in HORIZONS:
    # Variável dependente: y deslocado s períodos à frente
    # Para cada hex×t, o alvo é y no período t+s
    # → criamos coluna y_fwd_s por shift negativo
    df_est[f'y_fwd{s}'] = df_est.groupby('hex_id')['y'].shift(-s)

    # Remove linhas sem target (últimas s obs de cada hex)
    df_s = df_est[df_est[f'y_fwd{s}'].notna()].copy()
    df_s[f'y_fwd{s}'] = df_s[f'y_fwd{s}'].astype(int)

    # Splits temporais
    df_train = df_s[df_s['t'] <= T_TRAIN_END]
    df_test  = df_s[df_s['t'] == T_TEST - s]  # features em T, target em T+s

    X_train = df_train[ALL_FEATS].fillna(0)
    y_train = df_train[f'y_fwd{s}']
    X_test  = df_test[ALL_FEATS].fillna(0)
    y_test  = df_test[f'y_fwd{s}']

    if len(y_test) == 0 or y_test.nunique() < 2:
        print(f's={s}: teste sem variação suficiente, pulando')
        continue

    # Modelo logit L2 com class_weight='balanced' (lida com desbalanceamento)
    clf = LogisticRegression(
        penalty='l2', C=1.0, class_weight='balanced',
        solver='liblinear', max_iter=1000, random_state=42
    )
    clf.fit(X_train, y_train)
    models[s] = (clf, df_test.copy(), X_test, y_test)

    print(f's={s} | treino: {len(y_train):,} obs ({y_train.mean():.1%} pos) | teste: {len(y_test):,} obs ({y_test.mean():.1%} pos)')

    # Coleta coeficientes
    for feat, beta in zip(ALL_FEATS, clf.coef_[0]):
        coef_rows.append({'feature': feat, 'horizonte': s, 'beta': beta,
                          'odds_ratio': np.exp(beta)})

print('\nEstimação concluída.')

t_max = 260 (período mais recente)
Treino: t ≤ 256 | Validação: 257–259 | Teste: t = 260
Obs após descartar lag12 warmup: 38,844
s=1 | treino: 38,220 obs (24.6% pos) | teste: 156 obs (10.9% pos)
s=2 | treino: 38,220 obs (24.6% pos) | teste: 156 obs (10.9% pos)
s=4 | treino: 38,220 obs (24.7% pos) | teste: 156 obs (10.9% pos)

Estimação concluída.


In [123]:
# ── 3.3 Métricas de validação: AUC-ROC, AUC-PR, PAI, Moran's I ──────────────

def compute_pai(y_true, y_proba, top_pct=0.10):
    """Prediction Accuracy Index (Chainey 2008): captura nos top_pct hexes / top_pct."""
    n = len(y_true)
    n_top = max(1, int(n * top_pct))
    top_idx = np.argsort(y_proba)[::-1][:n_top]
    captured = y_true.iloc[top_idx].sum()
    total_events = y_true.sum()
    if total_events == 0:
        return 0.0
    return (captured / total_events) / top_pct

def compute_morans_i(residuals, hex_ids, fm_hex_set):
    """Moran's I dos resíduos médios por hex usando libpysal + esda."""
    mean_resid = pd.Series(residuals.values, index=hex_ids.values)
    mean_resid = mean_resid.groupby(mean_resid.index).mean()
    cells = list(mean_resid.index)

    # Matriz de pesos W baseada em vizinhos ring-1 dentro da FM
    w_dict = {}
    for cell in cells:
        neighbors = [nb for nb in set(h3.grid_disk(cell, 1)) - {cell} if nb in fm_hex_set and nb in mean_resid.index]
        w_dict[cell] = neighbors

    try:
        W = lw.W(w_dict, silence_warnings=True)
        mi = esda.Moran(mean_resid.loc[W.id_order].values, W)
        return mi.I, mi.p_sim
    except Exception as e:
        print(f'  Moran I erro: {e}')
        return np.nan, np.nan

metrics_rows = []
for s, (clf, df_test_s, X_test, y_test) in models.items():
    X_train = df_est[df_est['t'] <= T_TRAIN_END][ALL_FEATS].fillna(0)
    y_train = df_est[df_est['t'] <= T_TRAIN_END][f'y_fwd{s}'] if f'y_fwd{s}' in df_est.columns else pd.Series()

    proba = clf.predict_proba(X_test)[:, 1]
    residuals = y_test - proba

    auc_roc = roc_auc_score(y_test, proba)
    auc_pr  = average_precision_score(y_test, proba)
    pai     = compute_pai(y_test.reset_index(drop=True), proba)

    moran_i, moran_p = compute_morans_i(
        residuals.reset_index(drop=True),
        df_test_s['hex_id'].reset_index(drop=True),
        fm_hex_set
    )

    # Salvaguardas (spec Seção 9)
    if auc_pr < 0.10:
        print(f'⚠ s={s}: AUC-PR={auc_pr:.3f} < 0.10 → recomenda-se granularidade maior (res 8) ou horizonte ampliado')
    if not np.isnan(moran_i) and moran_i > 0.10 and moran_p < 0.01:
        print(f'⚠ s={s}: Moran I={moran_i:.3f} (p={moran_p:.3f}) > 0.10 → adicionar W²y ou efeitos fixos de RA')

    metrics_rows.append({
        'horizonte': s,
        'auc_roc': round(auc_roc, 4),
        'auc_pr': round(auc_pr, 4),
        'pai_top10': round(pai, 4),
        'morans_i': round(moran_i, 4) if not np.isnan(moran_i) else np.nan,
        'morans_p': round(moran_p, 4) if not np.isnan(moran_p) else np.nan,
        'n_train': len(X_train),
        'n_test': len(X_test),
        'taxa_positivos_train': round(float(y_train.mean()), 4) if len(y_train) > 0 else np.nan,
        'taxa_positivos_test': round(float(y_test.mean()), 4),
    })
    print(f's={s} | AUC-ROC={auc_roc:.3f} | AUC-PR={auc_pr:.3f} | PAI={pai:.2f} | Moran I={moran_i:.3f} (p={moran_p:.3f})')

df_metrics = pd.DataFrame(metrics_rows)
display(df_metrics)

s=1 | AUC-ROC=0.727 | AUC-PR=0.371 | PAI=3.53 | Moran I=0.135 (p=0.014)
s=2 | AUC-ROC=0.753 | AUC-PR=0.387 | PAI=3.53 | Moran I=0.117 (p=0.017)
⚠ s=4: Moran I=0.143 (p=0.005) > 0.10 → adicionar W²y ou efeitos fixos de RA
s=4 | AUC-ROC=0.709 | AUC-PR=0.379 | PAI=3.53 | Moran I=0.143 (p=0.005)


,horizonte,auc_roc,auc_pr,pai_top10,morans_i,morans_p,n_train,n_test,taxa_positivos_train,taxa_positivos_test
0,1,0.7275,0.3713,3.5294,0.1350,0.014,38220,156,0.2458,0.109
1,2,0.7529,0.3871,3.5294,0.1167,0.017,38220,156,0.2464,0.109
2,4,0.7093,0.3792,3.5294,0.1434,0.005,38220,156,0.2465,0.109


---
## Fase 4 — Predição com dados mais recentes

In [124]:
# ── 4.1 Predição para o período T (mais recente disponível) ──────────────────
# Features construídas em T; prediz probabilidade em T+s para cada hex.

df_T = df_full[df_full['t'] == T_TEST].copy()

print(f'Hexes na semana T (período {df_T["ano_sem"].iloc[0] if len(df_T) > 0 else "?"}): {len(df_T)}')

# Calcula atribuição top-3 drivers por hex
def top_drivers(coef_vector, feat_names, x_row, n=3):
    """Retorna os n features com maior |beta * x| (contribuição absoluta)."""
    contribs = np.abs(coef_vector * x_row)
    total = contribs.sum()
    idx = np.argsort(contribs)[::-1][:n]
    return [(feat_names[i], contribs[i] / total if total > 0 else 0) for i in idx]

pred_dfs = {}
for s, (clf, _, _, _) in models.items():
    X_T = df_T[ALL_FEATS].fillna(0)
    proba_s = clf.predict_proba(X_T)[:, 1]

    df_pred = df_T[['hex_id', 'area_fm', 'lat_centroide', 'lon_centroide', 'y_lag1']].copy()
    df_pred[f'prob_crime'] = proba_s

    # Decil de risco (dentro das áreas FM)
    df_pred['decil_risco'] = pd.qcut(proba_s, q=10, labels=False, duplicates='drop') + 1

    # Top-3 drivers por hex
    coef_vec = clf.coef_[0]
    driver_rows = []
    for x_arr in X_T.values:
        drivers = top_drivers(coef_vec, ALL_FEATS, x_arr, n=3)
        driver_rows.append({
            'top_driver_1': drivers[0][0] if len(drivers) > 0 else '',
            'contrib_top1': round(drivers[0][1], 4) if len(drivers) > 0 else 0,
            'top_driver_2': drivers[1][0] if len(drivers) > 1 else '',
            'contrib_top2': round(drivers[1][1], 4) if len(drivers) > 1 else 0,
            'top_driver_3': drivers[2][0] if len(drivers) > 2 else '',
            'contrib_top3': round(drivers[2][1], 4) if len(drivers) > 2 else 0,
        })
    df_drivers = pd.DataFrame(driver_rows)
    df_pred = pd.concat([df_pred.reset_index(drop=True), df_drivers], axis=1)

    # Adiciona todas as features (rastreabilidade)
    df_pred = pd.concat([df_pred, X_T.reset_index(drop=True)], axis=1)

    pred_dfs[s] = df_pred
    print(f's={s} | prob média: {proba_s.mean():.3f} | top decil (10): {(df_pred["decil_risco"] == 10).sum()} hexes')


# ── Agregação dos top drivers em nível de área FM ────────────────────────────
# Usada nos popups dos polígonos: combina os top_driver_{1,2,3} de cada hex
# da área, pondera pela P(crime) e devolve os N principais fatores em
# linguagem legível para policy maker.
def area_top_drivers_summary(df_area, n=5):
    """Agrega top drivers dos hexes em uma área FM, ponderando por P(crime).

    Retorna lista [(label_legível, share_relativa), ...] em ordem decrescente.
    share_relativa = fração da explicação total atribuída àquela feature na área.
    """
    if df_area is None or len(df_area) == 0:
        return []
    pieces = []
    for k in (1, 2, 3):
        d = df_area[[f"top_driver_{k}", f"contrib_top{k}", "prob_crime"]].copy()
        d.columns = ["feat", "contrib", "p"]
        d["score"] = d["contrib"].fillna(0) * d["p"].fillna(0)
        pieces.append(d)
    df_all = pd.concat(pieces, ignore_index=True)
    df_all = df_all[df_all["feat"].astype(str).str.len() > 0]
    if df_all.empty:
        return []
    agg = df_all.groupby("feat")["score"].sum().sort_values(ascending=False)
    total = agg.sum()
    if total <= 0:
        return []
    agg = agg / total
    return [(pretty_label(f), float(agg[f])) for f in agg.head(n).index]


def _drivers_html_block(df_area, title="Fatores que mais explicam o risco", n=5):
    """Bloco HTML compacto com os top-n drivers de uma área FM."""
    drivers = area_top_drivers_summary(df_area, n=n)
    if not drivers:
        items = "<li><i>sem hexes preditos nesta área</i></li>"
    else:
        items = "".join(
            f"<li>{lab} <span style='color:#888'>({sc:.0%})</span></li>"
            for lab, sc in drivers
        )
    return (
        f"<div style='margin-top:6px'>"
        f"<b style='font-size:11px'>{title}:</b>"
        f"<ol style='margin:4px 0 0 18px;padding:0;font-size:12px'>{items}</ol>"
        f"</div>"
    )


Hexes na semana T (período 2025-W01): 156
s=1 | prob média: 0.462 | top decil (10): 16 hexes
s=2 | prob média: 0.459 | top decil (10): 16 hexes
s=4 | prob média: 0.462 | top decil (10): 16 hexes


---
## Outputs obrigatórios

In [125]:
# ── Salva todos os CSVs de output ─────────────────────────────────────────────

# predicoes_t{s}.csv
for s, df_pred in pred_dfs.items():
    path = OUTPUT_DIR / f'predicoes_t{s}.csv'
    df_pred.to_csv(path, index=False)
    print(f'Salvo: {path} ({len(df_pred)} linhas)')

# coeficientes_logit.csv
df_coef_all = pd.DataFrame(coef_rows)
df_coef_pivot = df_coef_all.pivot(index='feature', columns='horizonte', values=['beta', 'odds_ratio'])
df_coef_pivot.columns = [f'{c[0]}_s{c[1]}' for c in df_coef_pivot.columns]
df_coef_pivot = df_coef_pivot.reset_index()
df_coef_pivot.to_csv(OUTPUT_DIR / 'coeficientes_logit.csv', index=False)
print(f'Salvo: output/coeficientes_logit.csv ({len(df_coef_pivot)} features)')

# metricas_validacao.csv
df_metrics.to_csv(OUTPUT_DIR / 'metricas_validacao.csv', index=False)
print(f'Salvo: output/metricas_validacao.csv')

print('\n=== MÉTRICAS FINAIS ===')
display(df_metrics)

Salvo: ..\output\predicoes_t1.csv (156 linhas)
Salvo: ..\output\predicoes_t2.csv (156 linhas)
Salvo: ..\output\predicoes_t4.csv (156 linhas)
Salvo: output/coeficientes_logit.csv (32 features)
Salvo: output/metricas_validacao.csv

=== MÉTRICAS FINAIS ===


,horizonte,auc_roc,auc_pr,pai_top10,morans_i,morans_p,n_train,n_test,taxa_positivos_train,taxa_positivos_test
0,1,0.7275,0.3713,3.5294,0.1350,0.014,38220,156,0.2458,0.109
1,2,0.7529,0.3871,3.5294,0.1167,0.017,38220,156,0.2464,0.109
2,4,0.7093,0.3792,3.5294,0.1434,0.005,38220,156,0.2465,0.109


In [126]:
# ── Exibe top-10 coeficientes por magnitude (s=1) ─────────────────────────────

if 1 in models:
    clf_s1 = models[1][0]
    coef_df = pd.DataFrame({'feature': ALL_FEATS, 'beta_s1': clf_s1.coef_[0]})
    coef_df['odds_ratio_s1'] = np.exp(coef_df['beta_s1'])
    coef_df['abs_beta'] = coef_df['beta_s1'].abs()
    print('Top 10 features por magnitude do coeficiente (s=1):')
    display(coef_df.nlargest(10, 'abs_beta')[['feature', 'beta_s1', 'odds_ratio_s1']].round(4))

Top 10 features por magnitude do coeficiente (s=1):


,feature,beta_s1,odds_ratio_s1
20,orcrim_CV,-1.7872,0.1674
24,fe_area_Campo Grande: Estação de Trem - Calçadão,-1.2550,0.2851
21,orcrim_Milícia,0.7737,2.1678
29,fe_area_Presidente Vargas - Campo de Santana -...,0.4018,1.4945
5,W_y_lag1,0.3653,1.4410
30,fe_area_Rodoviária - Terminal Gentileza - Esta...,-0.3649,0.6943
23,orcrim_TCP,-0.2937,0.7455
28,fe_area_Praia de Botafogo - Rua Marquês de Abr...,-0.2598,0.7712
26,fe_area_Jardim de Alah,-0.2496,0.7791
0,y_lag1,0.2346,1.2644


In [127]:
# ── Mapa de risco — predições T+1 sobre hexágonos FM ─────────────────────────

if 1 in pred_dfs:
    df_plot = pred_dfs[1].copy()

    m = folium.Map(location=[-22.92, -43.28], zoom_start=13, tiles="CartoDB dark_matter")

    # Colormap (amarelo → vermelho escuro) para risco dos hexes
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    cmap = plt.cm.YlOrRd

    prob_min = df_plot["prob_crime"].min()
    prob_max = df_plot["prob_crime"].max()

    # ─── Camada 1: contorno das áreas FM (polígonos reconstruídos via convex hull + buffer 120m) ───
    area_layer = folium.FeatureGroup(name="Áreas FM (contorno)", show=True)
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    ]
    for i, (area_name, poly) in enumerate(fm_polygons.items()):
        color = palette[i % len(palette)]
        exterior = list(poly.exterior.coords)
        locations = [(lat, lon) for lon, lat in exterior]

        # Popup enriquecido: resumo de risco da área + top fatores em linguagem natural
        df_area_pop = df_plot[df_plot["area_fm"] == area_name]
        n_hex = len(df_area_pop)
        p_med = df_area_pop["prob_crime"].mean() if n_hex else 0.0
        p_max = df_area_pop["prob_crime"].max() if n_hex else 0.0
        popup_html = (
            f"<div style='font-family:Arial,sans-serif;font-size:12px;min-width:240px'>"
            f"<b style='font-size:13px'>{area_name}</b><br>"
            f"<span style='color:#555;font-size:11px'>"
            f"{n_hex} hexes · P média: {p_med:.1%} · P máx: {p_max:.1%}"
            f"</span>"
            f"<hr style='margin:6px 0;border:none;border-top:1px solid #ddd'>"
            f"{_drivers_html_block(df_area_pop)}"
            f"</div>"
        )

        folium.Polygon(
            locations=locations,
            color=color,
            weight=2.5,
            fill=True,
            fill_color=color,
            fill_opacity=0.05,
            tooltip=f"Área FM: {area_name}",
            popup=folium.Popup(popup_html, max_width=360),
        ).add_to(area_layer)
    area_layer.add_to(m)

    # ─── Camada 2: hexágonos H3 coloridos pelo risco T+1 ───
    hex_layer = folium.FeatureGroup(name="Hexágonos FM — Risco T+1", show=True)
    for _, row in df_plot.iterrows():
        boundary = h3.cell_to_boundary(row["hex_id"])
        polygon_coords = [(lat, lon) for lat, lon in boundary]
        norm = (row["prob_crime"] - prob_min) / (prob_max - prob_min + 1e-9)
        color_hex = mcolors.to_hex(cmap(norm))

        # Traduz drivers técnicos em linguagem natural
        top1 = pretty_label(row.get("top_driver_1", ""))
        top2 = pretty_label(row.get("top_driver_2", ""))

        folium.Polygon(
            locations=polygon_coords,
            color=color_hex,
            weight=0.5,
            fill=True,
            fill_color=color_hex,
            fill_opacity=0.7,
            tooltip=(
                f"Área: {row['area_fm']}<br>"
                f"P(crime T+1): {row['prob_crime']:.1%}<br>"
                f"Decil risco: {row.get('decil_risco', 'n/a')}/10<br>"
                f"Fator 1: {top1}<br>"
                f"Fator 2: {top2}"
            ),
        ).add_to(hex_layer)
    hex_layer.add_to(m)

    # Legenda
    legend_html = """
    <div style="position:fixed;bottom:40px;right:20px;z-index:1000;
                background:rgba(20,20,20,0.92);color:white;
                padding:12px 16px;border-radius:8px;border:1px solid #555;
                font-family:Arial,sans-serif;font-size:12px;">
      <b>Risco de roubo — T+1</b><br><br>
      <span style="background:linear-gradient(to right,#ffffb2,#fd8d3c,#bd0026);
        display:inline-block;width:100px;height:10px;border-radius:3px"></span><br>
      Menor &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Maior<br><br>
      Unidade: hexágono H3 res 9 (~0,1 km²)<br>
      Modelo: Logit L2 — apenas áreas FM<br>
      Contornos coloridos: áreas da Força Municipal
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    folium.LayerControl(collapsed=False).add_to(m)

    m.save(str(OUTPUT_DIR / "mapa_risco_t1.html"))
    print("Mapa salvo → output/mapa_risco_t1.html")
    m


Mapa salvo → output/mapa_risco_t1.html


In [128]:
# ── Top hexes de maior risco por área FM + drivers explicativos ──────────────
# Para cada área da Força Municipal, identifica os N hexes com maior P(crime T+1)
# e lista os top 3 drivers (variáveis com maior contribuição |β·x|) desses hexes.

TOP_N_POR_AREA = 5  # quantos hexes mostrar por área

if 1 in pred_dfs:
    df_pred1 = pred_dfs[1].copy()

    # Ordena por probabilidade dentro de cada área e pega os top N
    df_top = (
        df_pred1.sort_values(["area_fm", "prob_crime"], ascending=[True, False])
                .groupby("area_fm", group_keys=False)
                .head(TOP_N_POR_AREA)
                .copy()
    )

    # Ranking dentro da área (1 = maior risco)
    df_top["rank_area"] = (
        df_top.groupby("area_fm")["prob_crime"]
              .rank(ascending=False, method="first")
              .astype(int)
    )

    # Aplica pretty_label nos drivers para apresentação ao policy maker
    df_top["driver_1_label"] = df_top["top_driver_1"].apply(pretty_label)
    df_top["driver_2_label"] = df_top["top_driver_2"].apply(pretty_label)
    df_top["driver_3_label"] = df_top["top_driver_3"].apply(pretty_label)

    cols_show = [
        "area_fm", "rank_area", "hex_id", "prob_crime", "decil_risco",
        "driver_1_label", "contrib_top1",
        "driver_2_label", "contrib_top2",
        "driver_3_label", "contrib_top3",
    ]
    df_top_show = df_top[cols_show].copy()
    df_top_show["prob_crime"]   = (df_top_show["prob_crime"]   * 100).round(1)
    df_top_show["contrib_top1"] = (df_top_show["contrib_top1"] * 100).round(1)
    df_top_show["contrib_top2"] = (df_top_show["contrib_top2"] * 100).round(1)
    df_top_show["contrib_top3"] = (df_top_show["contrib_top3"] * 100).round(1)
    df_top_show = df_top_show.rename(columns={
        "prob_crime":     "P(crime)%",
        "driver_1_label": "Driver 1",
        "contrib_top1":   "contrib1_%",
        "driver_2_label": "Driver 2",
        "contrib_top2":   "contrib2_%",
        "driver_3_label": "Driver 3",
        "contrib_top3":   "contrib3_%",
    })

    print(f"Top {TOP_N_POR_AREA} hexes por área FM (horizonte T+1):")
    display(df_top_show.set_index(["area_fm", "rank_area"]))

    # Salva CSV e LaTeX com rótulos legíveis
    df_top_show.to_csv(OUTPUT_DIR / "top_drivers_por_area.csv", index=False)
    try:
        df_top_show.to_latex(
            OUTPUT_DIR / "top_drivers_por_area.tex",
            index=False, float_format="%.1f"
        )
    except Exception as e:
        print(f"(LaTeX export pulado: {e})")

    print(f"\nSalvo: output/top_drivers_por_area.csv")
else:
    print("pred_dfs[1] indisponível — execute a Fase 4 antes")


Top 5 hexes por área FM (horizonte T+1):


hex_id  \
area_fm                                            rank_area                    
Bangu: Calçadão - Bangu Shopping                   1          89a8a0663cbffff   
                                                   2          89a8a066353ffff   
                                                   3          89a8a066227ffff   
                                                   4          89a8a0663cfffff   
                                                   5          89a8a066237ffff   
Campo Grande: Estação de Trem - Calçadão           1          89a8a0291d3ffff   
                                                   2          89a8a02956bffff   
                                                   3          89a8a0291d7ffff   
                                                   4          89a8a029567ffff   
                                                   5          89a8a029577ffff   
Estações São Francisco Xavier - Afonso Pena        1          89a8a061623ffff   
                                                   2          89a8a061677ffff   
                                                   3          89a8a061633ffff   
                                                   4          89a8a06160fffff   
                                                   5          89a8a061617ffff   
Jardim de Alah                                     1          89a8a07ab47ffff   
                                                   2          89a8a07ab73ffff   
                                                   3          89a8a07ab67ffff   
                                                   4          89a8a07ab0bffff   
                                                   5          89a8a07ab7bffff   
Metrô Botafogo - Rua São Clemente - Rua Voluntá... 1          89a8a07884fffff   
                                                   2          89a8a07884bffff   
                                                   3          89a8a078abbffff   
                                                   4          89a8a07880bffff   
                                                   5          89a8a078ab3ffff   
Praia de Botafogo - Rua Marquês de Abrantes        1          89a8a06a493ffff   
                                                   2          89a8a078a33ffff   
                                                   3          89a8a06a49bffff   
                                                   4          89a8a078aa3ffff   
                                                   5          89a8a078aabffff   
Presidente Vargas - Campo de Santana - Central ... 1          89a8a06a097ffff   
                                                   2          89a8a06a55bffff   
                                                   3          89a8a06a0a3ffff   
                                                   4          89a8a06a093ffff   
                                                   5          89a8a06a42fffff   
Rodoviária - Terminal Gentileza - Estação Leopo... 1          89a8a06a533ffff   
                                                   2          89a8a06a577ffff   
                                                   3          89a8a06a5afffff   
                                                   4          89a8a0612cfffff   
                                                   5          89a8a06a5b7ffff   
Rua Lauro Müller – Avenida General Severiano – ... 1          89a8a078a8bffff   
                                                   2          89a8a078a8fffff   
                                                   3          89a8a078a87ffff   
                                                   4          89a8a078a83ffff   
                                                   5          89a8a078a97ffff   

                                                              P(crime)%  \
area_fm                                            rank_area              
Bangu: Calçadão - Bangu Shopping                   1               54.3   
                                      


Salvo: output/top_drivers_por_area.csv


In [129]:
# ── Mapas individuais por área FM (HTML) ─────────────────────────────────────
# Gera um HTML separado para cada área da Força Municipal com:
#   • Contorno do polígono da área (linha grossa) com popup dos fatores explicativos
#   • Hexágonos H3 coloridos por P(crime T+1)
#   • Marcadores destacando os top hexes com drivers em LINGUAGEM NATURAL

import re
import unicodedata
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def slugify(text: str) -> str:
    """Converte nome da área em slug seguro para arquivo."""
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_str = nfkd.encode("ASCII", "ignore").decode("ASCII")
    slug = re.sub(r"[^a-zA-Z0-9]+", "_", ascii_str).strip("_").lower()
    return slug[:60]

if 1 in pred_dfs:
    df_pred1 = pred_dfs[1].copy()
    cmap = plt.cm.YlOrRd

    map_dir = OUTPUT_DIR / "mapas_por_area"
    map_dir.mkdir(exist_ok=True)

    print(f"Gerando mapas individuais em {map_dir}/...\n")

    saved = []
    for area_name, poly in fm_polygons.items():
        df_area = df_pred1[df_pred1["area_fm"] == area_name].copy()
        if df_area.empty:
            print(f"  [skip] {area_name[:60]}: sem hexes preditos")
            continue

        cx, cy = poly.centroid.x, poly.centroid.y
        m = folium.Map(location=[cy, cx], zoom_start=15, tiles="CartoDB dark_matter")

        p_min = df_area["prob_crime"].min()
        p_max = df_area["prob_crime"].max()

        # ─── Contorno do polígono da área com popup explicativo ───
        exterior = list(poly.exterior.coords)
        locations = [(lat, lon) for lon, lat in exterior]
        popup_html = (
            f"<div style='font-family:Arial,sans-serif;font-size:12px;min-width:240px'>"
            f"<b style='font-size:13px'>{area_name}</b><br>"
            f"<span style='color:#555;font-size:11px'>"
            f"{len(df_area)} hexes · P média: {df_area['prob_crime'].mean():.1%} · "
            f"P máx: {p_max:.1%}"
            f"</span>"
            f"<hr style='margin:6px 0;border:none;border-top:1px solid #ddd'>"
            f"{_drivers_html_block(df_area)}"
            f"</div>"
        )
        folium.Polygon(
            locations=locations,
            color="#4ecdc4",
            weight=3,
            fill=False,
            tooltip=f"Área FM: {area_name}",
            popup=folium.Popup(popup_html, max_width=360),
        ).add_to(m)

        # ─── Hexes coloridos por risco ───
        hex_layer = folium.FeatureGroup(name="Hexes — risco T+1", show=True)
        for _, row in df_area.iterrows():
            boundary = h3.cell_to_boundary(row["hex_id"])
            poly_coords = [(lat, lon) for lat, lon in boundary]
            norm = (row["prob_crime"] - p_min) / (p_max - p_min + 1e-9)
            color_hex = mcolors.to_hex(cmap(norm))
            folium.Polygon(
                locations=poly_coords,
                color=color_hex,
                weight=0.5,
                fill=True,
                fill_color=color_hex,
                fill_opacity=0.7,
                tooltip=f"P(crime T+1): {row['prob_crime']:.1%}",
            ).add_to(hex_layer)
        hex_layer.add_to(m)

        # ─── Marcadores nos top hexes (popup em linguagem natural) ───
        top_layer = folium.FeatureGroup(name=f"Top {TOP_N_POR_AREA} hexes — drivers", show=True)
        df_area_top = df_area.nlargest(TOP_N_POR_AREA, "prob_crime")
        for rank, (_, row) in enumerate(df_area_top.iterrows(), start=1):
            lat_c, lon_c = h3.cell_to_latlng(row["hex_id"])
            # Traduz cada driver técnico → texto legível
            d1 = pretty_label(row["top_driver_1"])
            d2 = pretty_label(row["top_driver_2"])
            d3 = pretty_label(row["top_driver_3"])
            popup_html = (
                f"<b>#{rank} — P(crime T+1): {row['prob_crime']:.1%}</b><br>"
                f"Decil: {int(row.get('decil_risco', 0))}/10<br><br>"
                f"<b>Fatores que mais explicam:</b><br>"
                f"1. {d1} ({row['contrib_top1']:.1%})<br>"
                f"2. {d2} ({row['contrib_top2']:.1%})<br>"
                f"3. {d3} ({row['contrib_top3']:.1%})"
            )
            folium.CircleMarker(
                location=[lat_c, lon_c],
                radius=10,
                color="#ffffff",
                weight=2,
                fill=True,
                fill_color="#bd0026",
                fill_opacity=0.95,
                tooltip=f"#{rank} — {row['prob_crime']:.1%}",
                popup=folium.Popup(popup_html, max_width=340),
            ).add_to(top_layer)
        top_layer.add_to(m)

        # Título no canto superior esquerdo
        title_html = (
            f'<div style="position:fixed;top:10px;left:50px;z-index:1000;'
            f'background:rgba(20,20,20,0.92);color:white;padding:8px 14px;'
            f'border-radius:6px;font-family:Arial;font-size:13px;max-width:60%;">'
            f'<b>{area_name}</b><br>Risco T+1 — Logit L2'
            f'</div>'
        )
        m.get_root().html.add_child(folium.Element(title_html))

        folium.LayerControl(collapsed=False).add_to(m)

        slug = slugify(area_name)
        fname = f"mapa_risco_t1_{slug}.html"
        m.save(str(map_dir / fname))
        saved.append(fname)
        print(f"  OK {fname}")

    print(f"\nTotal: {len(saved)} mapas salvos em output/mapas_por_area/")
else:
    print("pred_dfs[1] indisponível — execute a Fase 4 antes")


Gerando mapas individuais em ..\output\mapas_por_area/...

  OK mapa_risco_t1_bangu_calcadao_bangu_shopping.html
  OK mapa_risco_t1_campo_grande_estacao_de_trem_calcadao.html
  OK mapa_risco_t1_estacoes_sao_francisco_xavier_afonso_pena.html
  OK mapa_risco_t1_jardim_de_alah.html
  OK mapa_risco_t1_metro_botafogo_rua_sao_clemente_rua_voluntarios_da_patria.html
  OK mapa_risco_t1_praia_de_botafogo_rua_marques_de_abrantes.html
  OK mapa_risco_t1_presidente_vargas_campo_de_santana_central_do_brasil_cinelan.html
  OK mapa_risco_t1_rodoviaria_terminal_gentileza_estacao_leopoldina.html
  OK mapa_risco_t1_rua_lauro_muller_avenida_general_severiano_avenida_venceslau.html

Total: 9 mapas salvos em output/mapas_por_area/


In [130]:
# -- Mapa interativo: camada por horizonte + poligonos FM (LayerControl nativo) --
# Versao simplificada: sem painel customizado de filtros (que estava cobrindo
# o mapa no display inline do Jupyter). Use o controle nativo do folium no
# canto superior direito para alternar horizontes e ligar/desligar poligonos.

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from shapely.geometry import Point
from folium import Figure

cmap = plt.cm.YlOrRd

# Mapeia hex -> areas FM que o contem (apenas para enriquecer o tooltip)
hex_areas_map = {}
for _, hr in df_fm.iterrows():
    pt = Point(hr["lon_centroide"], hr["lat_centroide"])
    matching = [name for name, poly in fm_polygons.items() if poly.contains(pt)]
    if not matching:
        matching = [hr["area_fm"]]
    hex_areas_map[hr["hex_id"]] = matching

# Escala global de cores (compartilhada entre os horizontes)
all_probs = []
for s in HORIZONS:
    if s in pred_dfs:
        all_probs.extend(pred_dfs[s]["prob_crime"].tolist())
P_MIN, P_MAX = min(all_probs), max(all_probs)

# Mapa base com TileLayer explicito (tiles="CartoDB dark_matter" estava
# falhando em carregar - cinza no lugar do mapa).
m = folium.Map(location=[-22.92, -43.28], zoom_start=12, tiles=None)
folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
    name="OpenStreetMap",
    control=False,  # sempre visivel; nao aparece no LayerControl
).add_to(m)

# Uma FeatureGroup por horizonte; so a de s=1 fica visivel por padrao
for s in HORIZONS:
    if s not in pred_dfs:
        continue
    fg = folium.FeatureGroup(
        name=f"Hexes - horizonte T+{s}",
        show=(s == 1),
        overlay=True,  # overlay (checkbox); base layer escondia o tile
    )
    df_s = pred_dfs[s].set_index("hex_id")
    for hex_id, row in df_s.iterrows():
        boundary = h3.cell_to_boundary(hex_id)
        poly_coords = [(lat, lon) for lat, lon in boundary]
        norm = (row["prob_crime"] - P_MIN) / (P_MAX - P_MIN + 1e-9)
        color_hex = mcolors.to_hex(cmap(norm))
        areas_str = ", ".join(hex_areas_map.get(hex_id, ["?"]))
        d1 = pretty_label(row.get("top_driver_1", ""))
        d2 = pretty_label(row.get("top_driver_2", ""))
        d3 = pretty_label(row.get("top_driver_3", ""))
        tooltip_html = (
            f"<b>Area(s):</b> {areas_str}<br>"
            f"<b>Horizonte T+{s}</b><br>"
            f"<b>P(crime):</b> {row['prob_crime']:.1%}<br>"
            f"<b>Decil:</b> {int(row.get('decil_risco', 0))}/10<br><br>"
            f"<b>Fatores que mais explicam:</b><br>"
            f"1. {d1} ({row['contrib_top1']:.1%})<br>"
            f"2. {d2} ({row['contrib_top2']:.1%})<br>"
            f"3. {d3} ({row['contrib_top3']:.1%})"
        )
        folium.Polygon(
            locations=poly_coords,
            color=color_hex, weight=0.5,
            fill=True, fill_color=color_hex, fill_opacity=0.7,
            tooltip=folium.Tooltip(tooltip_html, sticky=True),
        ).add_to(fg)
    fg.add_to(m)

# Camada unica com todos os poligonos FM (visivel por padrao)
palette = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]
# Usa o horizonte s=1 (mais imediato) para o popup agregado dos poligonos
df_pop_base = pred_dfs[1] if 1 in pred_dfs else next(iter(pred_dfs.values()))
fg_polys = folium.FeatureGroup(name="Poligonos FM", show=True, overlay=True)
for i, (area_name, poly) in enumerate(fm_polygons.items()):
    color = palette[i % len(palette)]
    locations = [(lat, lon) for lon, lat in poly.exterior.coords]

    df_area_pop = df_pop_base[df_pop_base["area_fm"] == area_name]
    n_hex = len(df_area_pop)
    p_med = df_area_pop["prob_crime"].mean() if n_hex else 0.0
    p_max = df_area_pop["prob_crime"].max() if n_hex else 0.0
    popup_html = (
        f"<div style='font-family:Arial,sans-serif;font-size:12px;min-width:240px'>"
        f"<b style='font-size:13px'>{area_name}</b><br>"
        f"<span style='color:#555;font-size:11px'>"
        f"{n_hex} hexes - P media (T+1): {p_med:.1%} - P max: {p_max:.1%}"
        f"</span>"
        f"<hr style='margin:6px 0;border:none;border-top:1px solid #ddd'>"
        f"{_drivers_html_block(df_area_pop, title='Fatores que mais explicam o risco (T+1)')}"
        f"</div>"
    )

    folium.Polygon(
        locations=locations,
        color=color, weight=2.5,
        fill=True, fill_color=color, fill_opacity=0.04,
        tooltip=f"Area FM: {area_name}",
        popup=folium.Popup(popup_html, max_width=360),
    ).add_to(fg_polys)
fg_polys.add_to(m)

# Controle nativo do folium (canto superior direito, expandido)
folium.LayerControl(collapsed=False).add_to(m)

# Titulo e legenda compactos
title_html = """
<div style="position:absolute;top:10px;left:60px;z-index:1000;
            background:rgba(20,20,20,0.92);color:white;
            padding:8px 14px;border-radius:6px;border:1px solid #555;
            font-family:Arial,sans-serif;font-size:13px;
            box-shadow:0 2px 6px rgba(0,0,0,0.5);">
  <b>CompStat-Rio - Mapa de Risco</b>
</div>
"""
legend_html = """
<div style="position:absolute;bottom:30px;left:10px;z-index:1000;
            background:rgba(20,20,20,0.92);color:white;
            padding:10px 14px;border-radius:6px;border:1px solid #555;
            font-family:Arial,sans-serif;font-size:11px;
            box-shadow:0 2px 6px rgba(0,0,0,0.5);">
  <b>Risco de roubo - Logit L2</b><br>
  <span style="background:linear-gradient(to right,#ffffb2,#fd8d3c,#bd0026);
    display:inline-block;width:120px;height:10px;border-radius:3px;"></span><br>
  Menor &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Maior
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))
m.get_root().html.add_child(folium.Element(legend_html))

out_path = OUTPUT_DIR / "mapa_interativo.html"
m.save(str(out_path))
print(f"Mapa interativo salvo -> {out_path}")
print(f"  - Horizontes: {[s for s in HORIZONS if s in pred_dfs]}")
print(f"  - Areas FM (poligonos): {len(fm_polygons)}")

# Display inline com altura maior para o iframe nao espremer o mapa
fig = Figure(width="100%", height=700)
fig.add_child(m)
fig


Mapa interativo salvo -> ..\output\mapa_interativo.html
  - Horizontes: [1, 2, 4]
  - Areas FM (poligonos): 9
